# LightOnOCR-1B-1025
LightOnOCR-1B-1025 är en storskalig OCR-modell (~1B parametrar) för visuell-till-text-transkribering av dokument.  




In [1]:
!nvidia-smi

# Install vLLM with correct CUDA
!pip install -q vllm==0.11.2 pypdfium2 pillow requests pymupdf

Fri Jan  2 17:32:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             11W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Start vLLM server

In [2]:
import subprocess
import time
import requests

# Start vLLM server in background
process = subprocess.Popen(
    "python -m vllm.entrypoints.openai.api_server "
    "--model lightonai/LightOnOCR-1B-1025 "
    '--limit-mm-per-prompt \'{"image": 1}\' '
    "--port 8000 "
    "--gpu-memory-utilization 0.9",
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print("Starting vLLM server (loading model)...")
print("This takes 2-3 minutes...\n")

# Wait for server to be ready
for i in range(180):
    time.sleep(2)
    try:
        r = requests.get("http://localhost:8000/health", timeout=2)
        if r.status_code == 200:
            print(f"\n✓ Server ready after {i*2} seconds!")
            break
    except:
        if i % 15 == 0:
            print(f"  {i*2} seconds...")

# Check if server is running
try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    print(f" Server status: {r.status_code}")
except:
    print(" Server failed to start. Checking logs...")
    output = process.stdout.read().decode()
    print(output[-3000:])

Starting vLLM server (loading model)...
This takes 2-3 minutes...

  0 seconds...
  30 seconds...
  60 seconds...
  90 seconds...
  120 seconds...
  150 seconds...

✓ Server ready after 168 seconds!
 Server status: 200


### Upload PDF

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload() 
    pdf_path = list(uploaded.keys())[0]
    print("Using PDF:", pdf_path)

except ImportError:
    pdf_path = "data/benchmark/flexqube_arsredovisning_2022.pdf" 
    
    print(f"Running locally, using: {pdf_path}")

Upload your PDF:


Saving FlexQube.pdf to FlexQube.pdf
PDF uploaded: FlexQube.pdf


### PDF to images

In [4]:
import fitz  # PyMuPDF
from PIL import Image

def pdf_to_images(pdf_path: str, dpi: int = 200) -> list[Image.Image]:
    doc = fitz.open(pdf_path)
    images = []
    zoom = dpi / 72
    matrix = fitz.Matrix(zoom, zoom)

    for page_num in range(len(doc)):
        page = doc[page_num]
        pix = page.get_pixmap(matrix=matrix)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        images.append(img)
        print(f"  Page {page_num + 1}: {pix.width}x{pix.height}px")

    doc.close()
    return images

print(f"Converting {pdf_path} to images...")
pages = pdf_to_images(pdf_path, dpi=200)
print(f"Converted {len(pages)} pages")

Converting FlexQube.pdf to images...
  Page 1: 1654x2339px
  Page 2: 3308x2339px
  Page 3: 3308x2339px
  Page 4: 3308x2339px
  Page 5: 3308x2339px
  Page 6: 3308x2339px
  Page 7: 3308x2339px
  Page 8: 3308x2339px
  Page 9: 3308x2339px
  Page 10: 3308x2339px
  Page 11: 3308x2339px
  Page 12: 3308x2339px
  Page 13: 3308x2339px
  Page 14: 3308x2339px
  Page 15: 3308x2339px
  Page 16: 3308x2339px
  Page 17: 3308x2339px
  Page 18: 3308x2339px
  Page 19: 3308x2339px
  Page 20: 3308x2339px
  Page 21: 3308x2339px
  Page 22: 3308x2339px
  Page 23: 3308x2339px
  Page 24: 3308x2339px
  Page 25: 3308x2339px
  Page 26: 3308x2339px
  Page 27: 3308x2339px
  Page 28: 3308x2339px
  Page 29: 3308x2339px
  Page 30: 3308x2339px
  Page 31: 3308x2339px
  Page 32: 3308x2339px
  Page 33: 3308x2339px
  Page 34: 3308x2339px
  Page 35: 3308x2339px
  Page 36: 3308x2339px
  Page 37: 3308x2339px
  Page 38: 3308x2339px
  Page 39: 3308x2339px
  Page 40: 3308x2339px
  Page 41: 3308x2339px
  Page 42: 3308x2339px
  Page

### OCR function

In [5]:
import base64
import requests
import io

ENDPOINT = "http://localhost:8000/v1/chat/completions"
MODEL = "lightonai/LightOnOCR-1B-1025"

def ocr_single_image(image: Image.Image) -> str:
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')

    payload = {
        "model": MODEL,
        "messages": [{
            "role": "user",
            "content": [{
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{image_base64}"}
            }]
        }],
        "max_tokens": 4096,
        "temperature": 0.2,
    }

    response = requests.post(ENDPOINT, json=payload)
    return response.json()['choices'][0]['message']['content']

### Run OCR

In [6]:
print("Running LightOnOCR...")
results = []

for i, page in enumerate(pages):
    print(f"Processing page {i + 1}/{len(pages)}...", flush=True)
    text = ocr_single_image(page)
    results.append(text)

ocr_result = "\n\n---\n\n".join(results)
print(f"Done! ({len(ocr_result):,} chars)")

Running LightOnOCR...
Processing page 1/68...
Processing page 2/68...
Processing page 3/68...
Processing page 4/68...
Processing page 5/68...
Processing page 6/68...
Processing page 7/68...
Processing page 8/68...
Processing page 9/68...
Processing page 10/68...
Processing page 11/68...
Processing page 12/68...
Processing page 13/68...
Processing page 14/68...
Processing page 15/68...
Processing page 16/68...
Processing page 17/68...
Processing page 18/68...
Processing page 19/68...
Processing page 20/68...
Processing page 21/68...
Processing page 22/68...
Processing page 23/68...
Processing page 24/68...
Processing page 25/68...
Processing page 26/68...
Processing page 27/68...
Processing page 28/68...
Processing page 29/68...
Processing page 30/68...
Processing page 31/68...
Processing page 32/68...
Processing page 33/68...
Processing page 34/68...
Processing page 35/68...
Processing page 36/68...
Processing page 37/68...
Processing page 38/68...
Processing page 39/68...
Processing p

### Save results

In [7]:
output_path = "lightonocr_output.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(ocr_result)

print(f" Saved to: {output_path}")
files.download(output_path)

 Saved to: lightonocr_output.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# Preview
print("OCR Output Preview:")
print("="*50)
print(ocr_result[:6000])

OCR Output Preview:
# Årsredovisning 2022

![image](image_1.png)

---

## Cockpit

### Mission Planner

- **Wait 8 Seconds**
- **Go to Station Process 1**

---

**READY TO FOLLOW**

---

**FLEXOUBE**

---

# Innehåll

## ÖVERSIKT
- Vår historia ................................................................. 6
- Året i korthet ............................................................... 8
- Sammanfattande nyckeltal & KPI:er ........................................ 10
- VD har ordet ................................................................. 12

## STRATEGI
- Marknad och Trender ......................................................... 18
- Strategisk Modell ........................................................... 20
- Strategisk Riktning ........................................................ 22
- Kundbas ................................................................. 24
- Innovation och Produktutveckling .......................................... 26
- Djupdykning/Produ